In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# This ensures your plots show up directly inside the notebook
%matplotlib inline

In [4]:
!pip install --upgrade numexpr bottleneck

In [17]:
import os
print(os.getcwd())
print(os.listdir())

E:\Projects\bengaluru-housing-eda\notebooks
['.ipynb_checkpoints', '01_data_cleaning.ipynb']


In [27]:
from pathlib import Path


def find_project_root(marker_folder='data'):
    current = Path.cwd()
    for directory in [current, *current.parents]:
        if (directory / marker_folder).is_dir():
            return directory
    raise FileNotFoundError(
        f"Could not find a '{marker_folder}' folder in '{current}' or any parent folder. "
        f"Make sure Bengaluru_House_Data.csv is inside a 'data' folder in the project."
    )

project_root = find_project_root()
df = pd.read_csv(project_root / 'data' / 'Bengaluru_House_Data.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [ ]:
df.shape

In [ ]:
# Display the count of missing values for each column
df.isnull().sum()

In [ ]:
df2 = df.drop(['society', 'balcony', 'availability', 'area_type'], axis='columns')

In [ ]:
df3 = df2.dropna()

In [ ]:
print("New Shape:", df3.shape)
print("\nMissing values now:\n", df3.isnull().sum())

In [ ]:
# 1. Let's see all the unique and messy ways people entered the size
print("Unique values in 'size' column:")
print(df3['size'].unique())

# 2. Create a new column called 'bhk' by taking the string, splitting it by the space, and keeping the first number
df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))

# 3. Check the new column alongside the old one
df3[['size', 'bhk']].head()

In [ ]:
df3['bhk'].unique()

In [ ]:
# quick helper function to check if a value is a normal decimal number
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

# 2. Look at all the rows where 'total_sqft' is NOT a normal number
# (The '~' symbol means "NOT")
df3[~df3['total_sqft'].apply(is_float)].head(10)

In [ ]:
# 1. Define a function to convert the messy ranges into a single average number
def convert_sqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None   # We return None for the weird text ones to drop them later

# 2. Apply the function to create a new clean DataFrame
df4 = df3.copy()
df4['total_sqft'] = df4['total_sqft'].apply(convert_sqft_to_num)

# 3. Drop the few rows that returned 'None' and check the  work
df4 = df4.dropna()
print("Cleaned shape:", df4.shape)

In [ ]:
# 1. Create a fresh copy of the dataframe
df5 = df4.copy()

# 2. Calculate the price per square foot
df5['price_per_sqft'] = df5['price'] * 100000 / df5['total_sqft']

# 3. Display the first few rows to verify the new column
df5.head()

In [ ]:
# Strip any accidental hidden spaces from the location names
df5.location = df5.location.apply(lambda x: x.strip())

# Count exactly how many unique locations exist
location_stats = df5.groupby('location')['location'].agg('count').sort_values(ascending=False)
print("Total unique locations:", len(location_stats))

# Look at the top 10 most popular locations
location_stats.head(10)

In real estate, a standard rule of thumb is that a single bedroom requires a minimum of 300 square feet. If the total square footage divided by the number of bedrooms is less than 300, it is likely a data entry error or an extreme anomaly.

In [ ]:

print("Examples of sqft outliers:")
display(df5[df5.total_sqft / df5.bhk < 300].head())

# 2. Filter out these outliers using the ~ (NOT) operator
df6 = df5[~(df5.total_sqft / df5.bhk < 300)]

# 3. Check how many rows we have left
print("\nNew shape after removing sqft outliers:", df6.shape)


Next, we have to look at the price. Some properties might be sold for exceptionally high or low prices due to specific reasons we don't have data on (like a family discount or a luxury mansion).

To fix this, we will write a function that calculates the mean and standard deviation of the price_per_sqft for each specific location. It will then filter out any property that falls outside of one standard deviation from that location's average.

In [ ]:

def remove_pps_outliers(df):
    """
    Removes extreme price outliers based on mean and standard deviation per location.
    """
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st = np.std(subdf.price_per_sqft)
        # Keep properties within 1 standard deviation
        reduced_df = subdf[(subdf.price_per_sqft > (m - st)) & (subdf.price_per_sqft <= (m + st))]
        df_out = pd.concat([df_out, reduced_df], ignore_index=True)
    return df_out

df7 = remove_pps_outliers(df6)
print("Shape after price outliers removed:", df7.shape)

In standard residential properties, having more bathrooms than bedrooms plus two (e.g., a 2 BHK apartment with 5 bathrooms) indicates either a commercial property or a data error

In [ ]:
# 1. Inspect cases where bathroom count exceeds BHK + 2
display(df7[df7.bath > df7.bhk + 2].head())

# 2. Keep only valid listings
df8 = df7[df7.bath < df7.bhk + 2]

# 3. Drop columns that are no longer needed for training
# 'size' is replaced by 'bhk', and 'price_per_sqft' was strictly for outlier detection
df9 = df8.drop(['size', 'price_per_sqft'], axis='columns')
print("Shape after cleanup:", df9.shape)

One-Hot Encoding (location)Machine learning models cannot perform arithmetic directly on string text like "Whitefield" or "Electronic City". We convert each location into a binary column ($1$ if the property is in that location, $0$ otherwise).

In [ ]:
# 1. Create dummy variables (one-hot encoding) for each location
dummies = pd.get_dummies(df9.location, dtype=int)

# 2. Drop 'other' (if it exists) to prevent the dummy variable trap
# and concatenate with the main dataframe
df10 = pd.concat([
    df9.drop('location', axis='columns'), 
    dummies.drop('other', axis='columns', errors='ignore')
], axis='columns')

# 3. Check the final model-ready dataframe
print("Final dataset shape for model building:", df10.shape)
df10.head(3)

we must separate the independent variables (features) from the dependent variable (the price we want to predict). Then, we will use scikit-learn's train_test_split function, which randomly divides datasets into training and testing subsets. We will hold back 20% of the data to test the model on properties it has never seen.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. X is all independent variables (drop the price column)
X = df10.drop('price', axis='columns')

# 2. y is the dependent variable (the target price)
y = df10.price

# 3. Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

print("Training data shape:", X_train.shape)

Train the Baseline Linear Regression Model
Linear Regression is highly interpretable for real estate pricing. We will fit the model using the training set and score it using the testing set.

In [ ]:
from sklearn.linear_model import LinearRegression

# 1. Initialize the model
lr_clf = LinearRegression()

# 2. Train (fit) the model using the training data
lr_clf.fit(X_train, y_train)

# 3. Check the baseline accuracy (R-squared score) on the test data
print("Baseline Accuracy:", lr_clf.score(X_test, y_test))

A single test split can sometimes be lucky or unlucky. To ensure the model yields stable predictions across all data segments, we use cross-validation. The ShuffleSplit technique will first shuffle the data and then split it into training and testing sets. We can then call the cross_val_score helper function on our estimator to compute the performance across 5 different randomized folds.

In [ ]:
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

# 1. Create a ShuffleSplit cross-validator to randomize 5 folds
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

# 2. Evaluate the model across the 5 different splits
scores = cross_val_score(LinearRegression(), X, y, cv=cv)

print("Cross-Validation Scores:", scores)

Instead of training all of these manually, we can write a function that loops through a dictionary of models and parameters, tests them all using cross-validation, and returns a clean scoreboard.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
import pandas as pd

def find_best_model(X, y):
    # 1. Define the algorithms and the parameters we want to test
    algos = {
        'linear_regression': {
            'model': LinearRegression(),
            'params': {
                'fit_intercept': [True, False]
            }
        },
        'lasso': {
            'model': Lasso(),
            'params': {
                'alpha': [1, 2],
                'selection': ['random', 'cyclic']
            }
        },
        'decision_tree': {
            'model': DecisionTreeRegressor(),
            'params': {
                'criterion': ['squared_error', 'friedman_mse'],
                'splitter': ['best', 'random']
            }
        }
    }
    
    scores = []
    # 2. Use the same ShuffleSplit from yesterday for a fair comparison
    cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)
    
    # 3. Loop through the models, train them, and save the best scores
    for algo_name, config in algos.items():
        gs = GridSearchCV(config['model'], config['params'], cv=cv, return_train_score=False)
        gs.fit(X, y)
        scores.append({
            'model': algo_name,
            'best_score': gs.best_score_,
            'best_params': gs.best_params_
        })

    # 4. Return the results as a clean DataFrame
    return pd.DataFrame(scores, columns=['model', 'best_score', 'best_params'])

# Run the function!
find_best_model(X, y)

Since Linear Regression is our winner, we will use the lr_clf model you trained yesterday to start predicting actual prices.

Because we one-hot encoded our locations into hundreds of columns, we need a custom function to locate the correct column for a specific neighborhood and plug in the square footage, bathrooms, and bedrooms

In [ ]:
import numpy as np

def predict_price(location, sqft, bath, bhk):    
    # 1. Find the column index for the requested location
    loc_index = np.where(X.columns == location)[0][0]

    # 2. Create an array of zeros matching our total feature count
    x = np.zeros(len(X.columns))
    
    # 3. Insert the specific property details
    x[0] = sqft
    x[1] = bath
    x[2] = bhk
    
    # 4. Flip the specific location column to a 1
    if loc_index >= 0:
        x[loc_index] = 1

    # 5. Ask the trained model to predict the price and return it
    return lr_clf.predict([x])[0]

# Let's test it! Predict the price of a 1000 sqft, 2 bath, 2 BHK in '1st Phase JP Nagar'
print("Predicted Price (in Lakhs):", predict_price('1st Phase JP Nagar', 1000, 2, 2))